# 03 — Silver Transformations

Typed, deduplicated tables built from the bronze VARIANT payloads.

- `data:field::type` for scalar paths
- `from_json(data:field::string, '<schema>')` + `LATERAL VIEW EXPLODE` for arrays.
  See `rostr_schemas.py` for the canonical PySpark `StructType` definitions; the
  inline DDL strings in this notebook mirror those.
- **Natural-key primary keys** (rostr handles / composite tuples) — surrogate MD5 keys are introduced in gold (notebook 04). Databricks Delta requires FK references to point at an actual PK, so this keeps FK relationships valid.
- **`INSERT OVERWRITE`** for full-refresh idempotency (MERGE isn't supported through Databricks Connect).
- **PK / FK RELY** constraints + **liquid clustering** on company role.

| Silver table          | Grain                         | PK | Clustered |
|-----------------------|-------------------------------|----|-----------|
| `silver_artists`        | artist                       | `rostr_id` | — |
| `silver_companies`      | company                      | `rostr_id` | `(role)` |
| `silver_company_people` | company × person             | `(company_rostr_id, person_rostr_id)` | `(company_rostr_id)` |
| `silver_artist_company` | artist × company × role      | `(artist_handle, company_rostr_id, role)` | `(role)` |
| `silver_artist_person`  | artist × company × role × person | `(artist_handle, company_rostr_id, role, person_rostr_id)` | `(role)` |

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

In [ ]:
UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'rostr_music_industry')
BRONZE_SCHEMA = f'{UC_SCHEMA}_bronze'
SILVER_SCHEMA = f'{UC_SCHEMA}_silver'
B = f'{UC_CATALOG}.{BRONZE_SCHEMA}'
S = f'{UC_CATALOG}.{SILVER_SCHEMA}'
print(f'Bronze: {B}\nSilver: {S}')

In [ ]:
def add_pk(table: str, name: str, cols: list):
    cat, sch, tbl = table.split('.')
    if spark.sql(f"SELECT 1 FROM {cat}.information_schema.table_constraints WHERE table_schema='{sch}' AND table_name='{tbl}' AND constraint_type='PRIMARY KEY' AND constraint_name='{name}'").collect():
        return
    for c in cols:
        try:
            spark.sql(f'ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL')
        except Exception as e:
            if 'ALREADY_NOT_NULLABLE' not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    print(f'  + PK {name} on {table}')

def add_fk(table: str, name: str, cols: list, ref_table: str, ref_cols: list):
    cat, sch, tbl = table.split('.')
    if spark.sql(f"SELECT 1 FROM {cat}.information_schema.table_constraints WHERE table_schema='{sch}' AND table_name='{tbl}' AND constraint_type='FOREIGN KEY' AND constraint_name='{name}'").collect():
        return
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)}) RELY")
    print(f'  + FK {name} on {table}')

## silver_artists

One row per artist. Carries social metric snapshots (`sp/ig/fb/tt/yt/bit_followers`),
AI-generated bio fields (`ai_*`), and the rostr-canonical handle/uuid.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_artists (
        rostr_id           STRING NOT NULL,
        uuid               STRING,
        airtable_id        STRING,
        artist_name        STRING,
        artist_type        STRING,
        gender             STRING,
        age                INT,
        birth_date         DATE,
        location           STRING,
        genres             ARRAY<STRING>,
        claimed            BOOLEAN,
        spotify_url        STRING,  spotify_followers   BIGINT,
        instagram_url      STRING,  instagram_followers BIGINT,
        facebook_url       STRING,  facebook_followers  BIGINT,
        tiktok_url         STRING,  tiktok_followers    BIGINT,
        youtube_url        STRING,  youtube_subscribers BIGINT,
        bandsintown_url    STRING,  bandsintown_trackers BIGINT, bandsintown_on_tour BOOLEAN,
        ai_about_section   STRING,
        ai_real_name       STRING,
        ai_nationality     STRING,
        ai_origin_city     STRING,
        ai_origin_country  STRING,
        ai_career_start    STRING,
        ai_roles           ARRAY<STRING>,
        ai_descriptors     ARRAY<STRING>,
        wikipedia_url      STRING,
        official_website   STRING,
        profile_url        STRING,
        avatar_url         STRING,
        _ingestion_timestamp TIMESTAMP
    )
""")
spark.sql(f"""
    INSERT OVERWRITE {S}.silver_artists
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY data:rostrId::string ORDER BY _ingestion_timestamp DESC) AS rn
        FROM {B}.bronze_artists
    )
    SELECT
        data:rostrId::string, data:uuid::string, data:airtableId::string,
        data:name::string, data:artistType::string, data:gender::string,
        data:age::int,
        TRY_CAST(SUBSTRING(data:birthDate::string, 1, 10) AS DATE),
        data:location::string,
        FROM_JSON(data:genres::string, 'ARRAY<STRING>'),
        data:claimed::boolean,
        data:spUrl::string,  data:spMetric::bigint,
        data:igUrl::string,  data:igMetric::bigint,
        data:fbUrl::string,  data:fbMetric::bigint,
        data:ttUrl::string,  data:ttMetric::bigint,
        data:ytUrl::string,  data:ytMetric::bigint,
        data:bitUrl::string, data:bitMetric::bigint, data:bitOnTour::boolean,
        data:aiAboutSection::string, data:aiRealName::string, data:aiNationality::string,
        data:aiOriginCity::string, data:aiOriginCountry::string, data:aiCareerStartDate::string,
        FROM_JSON(data:aiRoles::string,       'ARRAY<STRING>'),
        FROM_JSON(data:aiDescriptors::string, 'ARRAY<STRING>'),
        data:aiWikipediaUrl::string, data:aiOfficialWebsiteUrl::string,
        data:profileUrl::string, data:avatarUrl::string,
        _ingestion_timestamp
    FROM ranked WHERE rn = 1 AND data:rostrId IS NOT NULL
""")
add_pk(f'{S}.silver_artists', 'silver_artists_pk', ['rostr_id'])
print('silver_artists:', spark.table(f'{S}.silver_artists').count())

## Bronze team → exploded company × team layer

`bronze_team` rows look like `{handle, role, entries:[{company, team:[{people:[...]}]}]}`.

We explode `entries` first → one row per (artist, company, role), then explode
`team[].people` for the artist-specific people. Two silver projections fall out
of the same exploded base, so we materialize it once as a temp view.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW v_bronze_team_exploded AS
    WITH base AS (
        SELECT
            data:handle::string AS artist_handle,
            data:role::string   AS role,
            entry, _ingestion_timestamp
        FROM {B}.bronze_team
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:entries::string,
                'ARRAY<STRUCT<'
                  'company:STRUCT<rostrId:STRING,uuid:STRING,airtableId:STRING,name:STRING,role:STRING,'
                                 'recordLabelType:STRING,formattedRecordLabelType:STRING,'
                                 'claimed:BOOLEAN,genres:ARRAY<STRING>,'
                                 'hqLocations:ARRAY<STRING>,otherLocations:ARRAY<STRING>,'
                                 'websiteUrl:STRING,igUrl:STRING,logoUrlLarge:STRING,'
                                 'profileUrl:STRING,radarDomain:STRING,radarEnabled:BOOLEAN,'
                                 'aiAboutSection:STRING,aiYearFounded:INT,'
                                 'parentCompany:STRING,parentCompanyProfileUrl:STRING,'
                                 'people:ARRAY<STRUCT<rostrId:STRING,airtableId:STRING,name:STRING,role:STRING,profileUrl:STRING>>>,'
                  'team:ARRAY<STRUCT<people:ARRAY<STRUCT<rostrId:STRING,uuid:STRING,airtableId:STRING,'
                                  'name:STRING,role:STRING,otherRoles:ARRAY<STRING>,'
                                  'execRoles:ARRAY<STRING>,genres:ARRAY<STRING>,'
                                  'email:STRING,phone:STRING,claimed:BOOLEAN,profileUrl:STRING,'
                                  'displayOrder:INT,companyRostrId:STRING,companyName:STRING,'
                                  'companyProfileUrl:STRING,companyLogoUrl:STRING>>>>'
                '>>'
            )
        ) AS entry
    )
    SELECT artist_handle, role, entry.company AS company, entry.team AS team, _ingestion_timestamp
    FROM base
""")
print('rows in temp view:', spark.sql('SELECT COUNT(*) FROM v_bronze_team_exploded').collect()[0][0])

## silver_companies

Deduped firm-level table — the same agency (e.g. WME) appears across many
artists; we keep the latest payload per `rostr_id`.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_companies (
        rostr_id               STRING NOT NULL,
        uuid                   STRING,
        airtable_id            STRING,
        company_name           STRING,
        role                   STRING,
        record_label_type      STRING,
        record_label_type_fmt  STRING,
        claimed                BOOLEAN,
        genres                 ARRAY<STRING>,
        hq_locations           ARRAY<STRING>,
        other_locations        ARRAY<STRING>,
        website_url            STRING,
        instagram_url          STRING,
        logo_url               STRING,
        profile_url            STRING,
        radar_domain           STRING,
        radar_enabled          BOOLEAN,
        ai_about_section       STRING,
        ai_year_founded        INT,
        parent_company         STRING,
        parent_company_profile_url STRING,
        _ingestion_timestamp   TIMESTAMP
    ) CLUSTER BY (role)
""")
spark.sql(f"""
    INSERT OVERWRITE {S}.silver_companies
    WITH all_companies AS (
        SELECT company.*, _ingestion_timestamp FROM v_bronze_team_exploded WHERE company.rostrId IS NOT NULL
    ),
    deduped AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY rostrId ORDER BY _ingestion_timestamp DESC) AS rn FROM all_companies
    )
    SELECT
        rostrId, uuid, airtableId, name, role, recordLabelType, formattedRecordLabelType, claimed,
        genres, hqLocations, otherLocations, websiteUrl, igUrl, logoUrlLarge, profileUrl,
        radarDomain, radarEnabled, aiAboutSection, aiYearFounded, parentCompany, parentCompanyProfileUrl,
        _ingestion_timestamp
    FROM deduped WHERE rn = 1
""")
add_pk(f'{S}.silver_companies', 'silver_companies_pk', ['rostr_id'])
print('silver_companies:', spark.table(f'{S}.silver_companies').count())

## silver_company_people

All employees we've seen at any company across the seed list. Composite
PK on (company_rostr_id, person_rostr_id).

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_company_people (
        company_rostr_id       STRING NOT NULL,
        person_rostr_id        STRING NOT NULL,
        airtable_id            STRING,
        person_name            STRING,
        person_role            STRING,
        profile_url            STRING,
        _ingestion_timestamp   TIMESTAMP
    ) CLUSTER BY (company_rostr_id)
""")
spark.sql(f"""
    INSERT OVERWRITE {S}.silver_company_people
    WITH exploded AS (
        SELECT company.rostrId AS company_rostr_id, p, _ingestion_timestamp
        FROM v_bronze_team_exploded
        LATERAL VIEW EXPLODE(company.people) AS p
        WHERE company.rostrId IS NOT NULL AND company.people IS NOT NULL
    ),
    deduped AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY company_rostr_id, p.rostrId ORDER BY _ingestion_timestamp DESC) AS rn FROM exploded
    )
    SELECT company_rostr_id, p.rostrId, p.airtableId, p.name, p.role, p.profileUrl, _ingestion_timestamp
    FROM deduped WHERE rn = 1 AND p.rostrId IS NOT NULL
""")
add_pk(f'{S}.silver_company_people', 'silver_company_people_pk', ['company_rostr_id', 'person_rostr_id'])
add_fk(f'{S}.silver_company_people', 'silver_company_people_company_fk',
       ['company_rostr_id'], f'{S}.silver_companies', ['rostr_id'])
print('silver_company_people:', spark.table(f'{S}.silver_company_people').count())

## silver_artist_company

M:N bridge — which firms represent which artists, and in what role.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_artist_company (
        artist_handle        STRING NOT NULL,
        company_rostr_id     STRING NOT NULL,
        role                 STRING NOT NULL,
        company_name         STRING,
        _ingestion_timestamp TIMESTAMP
    ) CLUSTER BY (role)
""")
spark.sql(f"""
    INSERT OVERWRITE {S}.silver_artist_company
    WITH base AS (
        SELECT artist_handle, role, company.rostrId AS company_rostr_id,
               company.name AS company_name, _ingestion_timestamp
        FROM v_bronze_team_exploded WHERE company.rostrId IS NOT NULL
    ),
    deduped AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY artist_handle, company_rostr_id, role ORDER BY _ingestion_timestamp DESC) AS rn FROM base
    )
    SELECT artist_handle, company_rostr_id, role, company_name, _ingestion_timestamp
    FROM deduped WHERE rn = 1
""")
add_pk(f'{S}.silver_artist_company', 'silver_artist_company_pk',
       ['artist_handle', 'company_rostr_id', 'role'])
add_fk(f'{S}.silver_artist_company', 'silver_artist_company_company_fk',
       ['company_rostr_id'], f'{S}.silver_companies', ['rostr_id'])
print('silver_artist_company:', spark.table(f'{S}.silver_artist_company').count())

## silver_artist_person

The richest grain — who is **specifically** assigned to each artist at each
company. Carries email/phone (when published).

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_artist_person (
        artist_handle       STRING NOT NULL,
        company_rostr_id    STRING NOT NULL,
        role                STRING NOT NULL,
        person_rostr_id     STRING NOT NULL,
        company_name        STRING,
        person_name         STRING,
        person_role         STRING,
        person_email        STRING,
        person_phone        STRING,
        exec_roles          ARRAY<STRING>,
        other_roles         ARRAY<STRING>,
        person_genres       ARRAY<STRING>,
        display_order       INT,
        person_profile_url  STRING,
        _ingestion_timestamp TIMESTAMP
    ) CLUSTER BY (role)
""")
spark.sql(f"""
    INSERT OVERWRITE {S}.silver_artist_person
    WITH base AS (
        SELECT artist_handle, role, company.rostrId AS company_rostr_id,
               company.name AS company_name, p, _ingestion_timestamp
        FROM v_bronze_team_exploded
        LATERAL VIEW EXPLODE(team) AS tg
        LATERAL VIEW EXPLODE(tg.people) AS p
        WHERE company.rostrId IS NOT NULL AND p.rostrId IS NOT NULL
    ),
    deduped AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY artist_handle, company_rostr_id, role, p.rostrId ORDER BY _ingestion_timestamp DESC) AS rn FROM base
    )
    SELECT artist_handle, company_rostr_id, role, p.rostrId, company_name,
           p.name, p.role, p.email, p.phone, p.execRoles, p.otherRoles, p.genres,
           p.displayOrder, p.profileUrl, _ingestion_timestamp
    FROM deduped WHERE rn = 1
""")
add_pk(f'{S}.silver_artist_person', 'silver_artist_person_pk',
       ['artist_handle', 'company_rostr_id', 'role', 'person_rostr_id'])
add_fk(f'{S}.silver_artist_person', 'silver_artist_person_company_fk',
       ['company_rostr_id'], f'{S}.silver_companies', ['rostr_id'])
print('silver_artist_person:', spark.table(f'{S}.silver_artist_person').count())

## Verify

In [ ]:
print('=' * 60)
print('SILVER LAYER SUMMARY')
print('=' * 60)
for t in ['silver_artists','silver_companies','silver_company_people',
          'silver_artist_company','silver_artist_person']:
    print(f'  {S}.{t:<35s} {spark.table(f"{S}.{t}").count():>8,} rows')

print('\nTop AGENCY companies by artist count:')
spark.sql(f"""
    SELECT company_name, COUNT(DISTINCT artist_handle) AS artist_count
    FROM {S}.silver_artist_company WHERE role = 'AGENCY'
    GROUP BY company_name ORDER BY artist_count DESC LIMIT 10
""").show(truncate=False)